# MOD-04 · NB-04 — Clinical Model Evaluation
### Health Analytics with Python · Module 04: Machine Learning for Clinical Prediction
---
**Learning objectives**
- Choose the right evaluation metric for imbalanced clinical outcomes
- Interpret AUC-ROC, Precision-Recall, Brier score, and net benefit together
- Plot and interpret calibration curves; apply Platt scaling and isotonic regression
- Compute Decision Curve Analysis (DCA) and net benefit at clinical thresholds
- Build a comprehensive evaluation report with confidence intervals

**Estimated time:** 2.5 hours | **Level:** Advanced | **Libraries:** `sklearn`, `scipy`, `matplotlib`


## 1. Setup, Dataset & Models

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                              roc_curve, precision_recall_curve, confusion_matrix,
                              classification_report)
import warnings; warnings.filterwarnings("ignore")
import os; os.makedirs("/tmp/mod04",exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})

np.random.seed(42); N=3000
def sig(x): return 1/(1+np.exp(-x))
b=np.random.normal(size=(N,4)); np.random.seed(99)
age=np.random.normal(62,15,N).clip(18,92).astype(int)
sex=np.random.choice(["M","F"],N,p=[0.48,0.52])
payer=np.random.choice(["Medicare","Medicaid","Private","Self-pay","Other"],N,p=[0.40,0.22,0.28,0.07,0.03])
admit_type=np.random.choice(["Emergency","Elective","Urgent"],N,p=[0.52,0.30,0.18])
los_days=np.random.gamma(2.5,1.8,N).clip(1,30).astype(int)
diabetes=np.random.binomial(1,sig(0.6*b[:,0]-0.5+0.02*(age-62)/15),N)
hypertension=np.random.binomial(1,sig(0.7*b[:,0]-0.2),N)
hf=np.random.binomial(1,sig(0.4*b[:,1]+0.5*hypertension-1.5),N)
copd=np.random.binomial(1,sig(0.5*b[:,2]-1.0),N)
ckd=np.random.binomial(1,sig(0.5*b[:,0]+0.6*diabetes-1.2),N)
obesity=np.random.binomial(1,sig(0.5*b[:,0]-0.8),N)
depression=np.random.binomial(1,sig(0.3*b[:,3]-1.4),N)
n_comorb=diabetes+hypertension+hf+copd+ckd
np.random.seed(21)
glucose=np.random.normal(105+15*diabetes,28,N).clip(50,400).round(1)
creatinine=np.random.gamma(1.6+0.8*hf,0.6,N).clip(0.4,10).round(2)
hba1c=np.random.normal(6.5+0.8*diabetes,1.4,N).clip(4,14).round(1)
sbp=np.random.normal(128+12*hypertension,18,N).clip(80,200).round(0)
bmi=np.random.normal(28+4*obesity,6,N).clip(15,55).round(1)
logit=(-3.2+0.6*hf+0.5*diabetes+0.5*ckd+0.3*copd+0.02*(age-62)/15
       +0.3*(admit_type=="Emergency").astype(int)+0.25*(payer=="Medicaid").astype(int)
       +0.2*(los_days>7).astype(int)+0.5*hf*diabetes+np.random.normal(0,0.25,N))
readmitted=np.random.binomial(1,sig(logit),N)
df=pd.DataFrame({
    "age":age,"sex":sex,"payer":payer,"admit_type":admit_type,"los_days":los_days,
    "diabetes":diabetes,"hypertension":hypertension,"hf":hf,"copd":copd,"ckd":ckd,
    "obesity":obesity,"depression":depression,"n_comorb":n_comorb,
    "glucose":glucose,"creatinine":creatinine,"hba1c":hba1c,"sbp":sbp,"bmi":bmi,"readmitted":readmitted,
})
df["los_gt7"]=(df.los_days>7).astype(int)
NUMERIC=["age","los_days","n_comorb","glucose","creatinine","hba1c","sbp","bmi"]
BINARY=["diabetes","hypertension","hf","copd","ckd","obesity","depression","los_gt7"]
CATEGORIC=["sex","payer","admit_type"]
FEATURES=NUMERIC+BINARY+CATEGORIC; TARGET="readmitted"
pre=ColumnTransformer([
    ("num",Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler())]),NUMERIC),
    ("bin",SimpleImputer(strategy="most_frequent"),BINARY),
    ("cat",Pipeline([("imp",SimpleImputer(strategy="most_frequent")),
                     ("ohe",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]),CATEGORIC),
])
X=df[FEATURES]; y=df[TARGET]
X_tr,X_te,y_tr,y_te=train_test_split(X,y,test_size=0.20,stratify=y,random_state=42)
X_tr2,X_cal,y_tr2,y_cal=train_test_split(X_tr,y_tr,test_size=0.20,stratify=y_tr,random_state=42)
pre.fit(X_tr2)
Xtr=pre.transform(X_tr2); Xcal=pre.transform(X_cal); Xte=pre.transform(X_te)

models = {
    "Logistic Reg" : LogisticRegression(C=1,max_iter=1000,class_weight="balanced",random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200,max_depth=10,class_weight="balanced",random_state=42,n_jobs=-1),
    "Grad Boosting": GradientBoostingClassifier(n_estimators=200,max_depth=4,learning_rate=0.05,subsample=0.8,random_state=42),
}
fitted = {}; probs = {}
for name,m in models.items():
    m.fit(Xtr,y_tr2); fitted[name]=m; probs[name]=m.predict_proba(Xte)[:,1]
print("Models trained. Positive rate (test):", y_te.mean()*100, "%")


## 2. AUC-ROC and Precision-Recall with Bootstrap CIs

In [ ]:
def bootstrap_ci(y_true, y_prob, metric_fn, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    scores = [metric_fn(y_true[idx:=rng.choice(len(y_true),len(y_true),replace=True)],
                        y_prob[idx]) for _ in range(n_boot)]
    return np.percentile(scores,[2.5,97.5])

fig,axes=plt.subplots(1,2,figsize=(15,6))
colors_m={"Logistic Reg":"#6ACC65","Random Forest":"#4878CF","Grad Boosting":"#D65F5F"}
summary=[]

for name,prob in probs.items():
    auc  = roc_auc_score(y_te,prob)
    ap   = average_precision_score(y_te,prob)
    brier= brier_score_loss(y_te,prob)
    auc_ci = bootstrap_ci(y_te.values,prob,roc_auc_score)
    ap_ci  = bootstrap_ci(y_te.values,prob,average_precision_score)
    summary.append({"Model":name,"AUC":auc,"AUC 95%CI":f"{auc_ci[0]:.3f}–{auc_ci[1]:.3f}",
                    "Avg Prec":ap,"AP 95%CI":f"{ap_ci[0]:.3f}–{ap_ci[1]:.3f}","Brier":brier})
    fpr,tpr,_=roc_curve(y_te,prob)
    prec,rec,_=precision_recall_curve(y_te,prob)
    axes[0].plot(fpr,tpr,lw=2,color=colors_m[name],
                 label=f"{name} AUC={auc:.3f} ({auc_ci[0]:.3f}–{auc_ci[1]:.3f})")
    axes[1].plot(rec,prec,lw=2,color=colors_m[name],
                 label=f"{name} AP={ap:.3f} ({ap_ci[0]:.3f}–{ap_ci[1]:.3f})")

axes[0].plot([0,1],[0,1],"k--",lw=1); axes[0].set_xlabel("1-Specificity"); axes[0].set_ylabel("Sensitivity")
axes[0].set_title("ROC Curves with Bootstrap 95% CI"); axes[0].legend(fontsize=8)
axes[1].axhline(y_te.mean(),color="red",ls="--",lw=1,label=f"No-skill ({y_te.mean():.2f})")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curves"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("/tmp/mod04/roc_pr_curves.png",bbox_inches="tight"); plt.show()
display(pd.DataFrame(summary).round(4))


## 3. Calibration Assessment & Recalibration

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(16,5))
fig.suptitle("Calibration plots — before and after recalibration",fontsize=12)

for i,(name,prob) in enumerate(probs.items()):
    ax=axes[i]
    # Original calibration
    frac_pos,mean_pred=calibration_curve(y_te,prob,n_bins=10)
    ax.plot(mean_pred,frac_pos,"o-",color=colors_m[name],lw=2,ms=5,label="Uncalibrated")
    # Platt scaling (sigmoid)
    lr_cal=LogisticRegression(C=1e6,max_iter=1000)
    lr_cal.fit(prob.reshape(-1,1),y_te)
    prob_platt=lr_cal.predict_proba(prob.reshape(-1,1))[:,1]
    fp2,mp2=calibration_curve(y_te,prob_platt,n_bins=10)
    ax.plot(mp2,fp2,"s--",color="orange",lw=1.5,ms=5,label="Platt (sigmoid)")
    # Isotonic regression
    from sklearn.isotonic import IsotonicRegression
    iso=IsotonicRegression(out_of_bounds="clip")
    iso.fit(prob,y_te)
    prob_iso=iso.predict(prob)
    fp3,mp3=calibration_curve(y_te,prob_iso,n_bins=10)
    ax.plot(mp3,fp3,"^:",color="purple",lw=1.5,ms=5,label="Isotonic")
    ax.plot([0,1],[0,1],"k--",lw=1,label="Perfect")
    brier_orig=brier_score_loss(y_te,prob)
    brier_platt=brier_score_loss(y_te,prob_platt)
    ax.text(0.05,0.92,f"Brier: {brier_orig:.3f}→{brier_platt:.3f}",
            transform=ax.transAxes,fontsize=9,bbox=dict(facecolor="white",alpha=0.8,edgecolor="none"))
    ax.set_xlabel("Mean predicted"); ax.set_ylabel("Fraction positives")
    ax.set_title(name); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig("/tmp/mod04/calibration.png",bbox_inches="tight"); plt.show()


## 4. Decision Curve Analysis (DCA) — Net Benefit

In [ ]:
def net_benefit(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tp = ((y_pred==1)&(y_true==1)).sum()
    fp = ((y_pred==1)&(y_true==0)).sum()
    n  = len(y_true)
    odds = threshold/(1-threshold)
    return (tp - fp*odds) / n

thresholds = np.linspace(0.01,0.50,100)
fig,ax=plt.subplots(figsize=(10,6))

for name,prob in probs.items():
    nb_vals=[net_benefit(y_te.values,prob,t) for t in thresholds]
    ax.plot(thresholds,nb_vals,lw=2,color=colors_m[name],label=name)

# Treat all (net benefit of treating everyone)
nb_all=[net_benefit(y_te.values,np.ones_like(y_te.values,dtype=float),t) for t in thresholds]
ax.plot(thresholds,nb_all,"k-.",lw=1.5,label="Treat all")
ax.axhline(0,color="gray",ls="--",lw=1,label="Treat none")
ax.fill_between(thresholds,0,
                np.maximum([max(0,v) for v in nb_all],0),alpha=0.05,color="gray")
ax.set_xlabel("Threshold probability"); ax.set_ylabel("Net benefit per patient")
ax.set_title("Decision Curve Analysis
(area above 'Treat all' and 'Treat none' = clinical utility)")
ax.legend(fontsize=9); ax.set_xlim(0,0.50); ax.set_ylim(-0.05,0.15)
plt.tight_layout(); plt.savefig("/tmp/mod04/dca.png",bbox_inches="tight"); plt.show()
print("DCA interpretation:")
print("  Treat all  = refer every patient regardless of model")
print("  Treat none = refer no patient")
print("  Model curve above both lines at a given threshold = model adds clinical value")
print("  Choose threshold that matches your clinical tradeoff (e.g. intervention cost vs harm of missing)")


## 5. Threshold Selection for Clinical Deployment

In [ ]:
best_prob = probs["Grad Boosting"]
fpr,tpr,thrs=roc_curve(y_te,best_prob)
precision_v,recall_v,thrs_pr=precision_recall_curve(y_te,best_prob)

# Three threshold strategies
thresholds_clinical={
    "Youden's J (balanced)": thrs[np.argmax(tpr-fpr)],
    "Sensitivity≥0.80"      : thrs[np.where(tpr>=0.80)[0][0]] if any(tpr>=0.80) else 0.5,
    "Precision≥0.40"        : thrs_pr[np.where(precision_v>=0.40)[0][0]] if any(precision_v>=0.40) else 0.5,
}
print(f"{'Strategy':30s} {'Threshold':>10s} {'Sens':>8s} {'Spec':>8s} {'PPV':>8s} {'NPV':>8s}")
print("-"*70)
for strategy, thr in thresholds_clinical.items():
    y_pred=(best_prob>=thr).astype(int)
    tn,fp,fn,tp=confusion_matrix(y_te,y_pred).ravel()
    sens=tp/(tp+fn); spec=tn/(tn+fp)
    ppv =tp/(tp+fp) if tp+fp>0 else 0
    npv =tn/(tn+fn) if tn+fn>0 else 0
    print(f"{strategy:30s} {thr:>10.3f} {sens:>8.3f} {spec:>8.3f} {ppv:>8.3f} {npv:>8.3f}")

print()
print("Clinical guidance:")
print("  High sensitivity = fewer missed readmissions (prefer when intervention is cheap/safe)")
print("  High precision   = fewer unnecessary interventions (prefer when intervention is costly/harmful)")


## 6. Knowledge Check
1. AUC=0.78 but Avg Precision=0.32. Why the large gap, and which is more informative here?
2. The calibration plot shows the RF overestimates risk at high probabilities. What recalibration approach should you use?
3. At threshold=0.20, GB model has net benefit > treat all. What does this mean clinically?
4. A colleague insists on maximising AUC. Name two scenarios where this would be the wrong objective.
5. Apply isotonic regression recalibration on the validation set and re-evaluate Brier score on the test set.


In [ ]:
# Exercise 5 — isotonic recalibration on val → test
from sklearn.isotonic import IsotonicRegression
for name,model in fitted.items():
    prob_val=model.predict_proba(Xcal)[:,1]
    prob_test=probs[name]
    iso=IsotonicRegression(out_of_bounds="clip")
    iso.fit(prob_val, y_cal.values)
    prob_recal=iso.predict(prob_test)
    brier_orig=brier_score_loss(y_te,prob_test)
    brier_recal=brier_score_loss(y_te,prob_recal)
    print(f"{name:20s}: Brier {brier_orig:.4f} → {brier_recal:.4f} (change: {brier_recal-brier_orig:+.4f})")


---
**Next → NB-05: Hyperparameter Tuning with Optuna**